# Simulation Results Analysis: Count Data Conformal Prediction

Analysis of simulation results for conformal prediction methods on count data.
Results are loaded from `results/` and figures saved to `analysis_output/` relative to the project root.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

from cp_claims.config import (
    BENCHMARK_METHODS, CP_METHODS, DEFAULT_MAIN_METHODS,
    DEFAULT_BASELINE, METHOD_DISPLAY_NAMES,
    DGCP_METHOD_TYPES, MIN_GROUP_SIZES, LEARNERS,
)
from cp_claims.analysis import (
    load_results, filter_methods, apply_baseline_filter,
    compute_summary_stats, compute_method_rankings,
    infer_baseline, get_display_name, summarize_results,
    generate_latex_table,
)

RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR = PROJECT_ROOT / 'analysis_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DGCP_METHODS = [m for m in DEFAULT_MAIN_METHODS if m.startswith('DGCP_')]

TITLE_SIZE = 14
LABEL_SIZE = 12
TICK_SIZE = 11
LEGEND_SIZE = 12


## 2. Load Simulation Results

In [ ]:
df = load_results(RESULTS_DIR)
dataset_info = summarize_results(df)

print(f'Loaded {dataset_info["rows"]:,} rows from {RESULTS_DIR}')
for col in ['dgp_type', 'learner', 'method', 'n_obs', 'dim_x',
             'correlation_strength', 'categorical_ratio', 'zero_inflation']:
    if col in dataset_info:
        print(f'  {col}: {dataset_info[col]}')


## 3. Configuration and Filtering

In [ ]:
df_full = df.copy()
df = filter_methods(df, DEFAULT_MAIN_METHODS)

BASELINE = infer_baseline(df, DEFAULT_BASELINE)
print('Baseline configuration:')
for param, value in BASELINE.items():
    print(f'  {param}: {value}')

df_baseline = apply_baseline_filter(df, BASELINE)
print(f'Baseline rows: {len(df_baseline):,} of {len(df):,}')

if df_baseline.empty and 'n_obs' in BASELINE:
    df_baseline = apply_baseline_filter(df, {'n_obs': BASELINE['n_obs']})
    print(f'Fallback (n_obs only): {len(df_baseline):,} rows')

if not df_baseline.empty:
    print(f'  DGP types: {df_baseline["dgp_type"].unique().tolist()}')
    print(f'  Learners: {df_baseline["learner"].unique().tolist()}')


## 4. Summary Statistics

In [ ]:
method_summary = df_baseline.groupby('method').agg({
    'coverage': ['mean', 'std', 'min', 'max'],
    'mean_width': ['mean', 'std'],
}).round(4)
display(method_summary)

valid_rate = (df_baseline['coverage'] >= 0.88).mean() * 100
print(f'Validity rate (coverage >= 88%): {valid_rate:.1f}%')


In [ ]:
summary_stats = compute_summary_stats(df_baseline)
summary_stats.to_csv(OUTPUT_DIR / 'summary_statistics.csv', index=False)
display(summary_stats.head(20))


## 5. Coverage Analysis by DGP

In [ ]:
dgps = df_baseline['dgp_type'].unique()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, dgp in enumerate(dgps[:4]):
    ax = axes[idx]
    subset = df_baseline[df_baseline['dgp_type'] == dgp]
    method_cov = subset.groupby('method')['coverage'].agg(['mean', 'std']).reset_index()
    method_cov = method_cov.sort_values('mean', ascending=True)
    method_cov['display_name'] = method_cov['method'].apply(get_display_name)

    colors = ['green' if m >= 0.88 else 'red' for m in method_cov['mean']]
    ax.barh(method_cov['display_name'], method_cov['mean'],
            xerr=method_cov['std'], color=colors, alpha=0.7, capsize=3)
    ax.axvline(0.90, color='black', linestyle='--', linewidth=2, label='Target (90%)')
    ax.axvline(0.88, color='gray', linestyle=':', linewidth=1.5, label='Tolerance (88%)')
    ax.set_xlabel('Coverage')
    ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
    ax.set_xlim(0.7, 1.0)
    ax.tick_params(axis='both', labelsize=TICK_SIZE)
    if idx == 0:
        ax.legend(loc='lower right', fontsize=TICK_SIZE)

for idx in range(len(dgps), 4):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coverage_by_dgp.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Coverage Analysis by Learner

In [ ]:
df_cp_only = df_baseline[~df_baseline['method'].isin(BENCHMARK_METHODS)]
learners = df_cp_only['learner'].unique()
n_learners = len(learners)

fig, axes = plt.subplots(1, max(1, n_learners), figsize=(6 * max(1, n_learners), 6))
if n_learners == 1:
    axes = [axes]

for idx, learner in enumerate(learners):
    ax = axes[idx]
    subset = df_cp_only[df_cp_only['learner'] == learner]
    method_cov = subset.groupby('method')['coverage'].agg(['mean', 'std']).reset_index()
    method_cov = method_cov.sort_values('mean', ascending=True)
    method_cov['display_name'] = method_cov['method'].apply(get_display_name)

    colors = ['green' if m >= 0.88 else 'red' for m in method_cov['mean']]
    ax.barh(method_cov['display_name'], method_cov['mean'],
            xerr=method_cov['std'], color=colors, alpha=0.7, capsize=3)
    ax.axvline(0.90, color='black', linestyle='--', linewidth=2)
    ax.axvline(0.88, color='gray', linestyle=':', linewidth=1.5)
    ax.set_xlabel('Coverage')
    ax.set_title(f'Learner: {learner}', fontsize=TITLE_SIZE, fontweight='bold')
    ax.set_xlim(0.7, 1.0)
    ax.tick_params(axis='both', labelsize=TICK_SIZE)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coverage_by_learner_cp_only.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Coverage vs Width Trade-off

In [ ]:
summary = df_baseline.groupby(['method', 'learner']).agg({
    'coverage': 'mean',
    'mean_width': 'mean',
}).reset_index()
summary['display_name'] = summary['method'].apply(get_display_name)

fig, ax = plt.subplots(figsize=(12, 8))
palette = sns.color_palette('colorblind', n_colors=summary['learner'].nunique())
learner_colors = dict(zip(summary['learner'].unique(), palette))

for learner in summary['learner'].unique():
    subset = summary[summary['learner'] == learner]
    ax.scatter(subset['mean_width'], subset['coverage'],
               label=learner, s=150, alpha=0.7,
               c=[learner_colors[learner]], edgecolors='black')

ax.axhline(0.90, color='red', linestyle='--', linewidth=2, label='Target Coverage')
ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5, label='Tolerance')
ax.set_xlabel('Mean Interval Width')
ax.set_ylabel('Coverage')
ax.set_title('Coverage vs Width Trade-off', fontsize=TITLE_SIZE, fontweight='bold')
ax.tick_params(axis='both', labelsize=TICK_SIZE)
ax.grid(alpha=0.3)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=len(labels), fontsize=LEGEND_SIZE)

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.savefig(OUTPUT_DIR / 'coverage_width_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Per-learner scatter with method-level colours
all_methods = summary['method'].unique()
method_palette = sns.color_palette('husl', n_colors=len(all_methods))
method_colors = dict(zip(all_methods, method_palette))

for learner in summary['learner'].unique():
    fig, ax = plt.subplots(figsize=(12, 9))
    subset = summary[summary['learner'] == learner]

    for _, row in subset.iterrows():
        ax.scatter(row['mean_width'], row['coverage'],
                   s=200, alpha=0.8,
                   c=[method_colors[row['method']]],
                   edgecolors='black', linewidths=1.5)

    ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
    ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
    ax.set_xlabel('Mean Interval Width')
    ax.set_ylabel('Coverage')
    ax.set_title(f'Coverage vs Width: {learner}', fontsize=TITLE_SIZE, fontweight='bold')
    ax.tick_params(axis='both', labelsize=TICK_SIZE)
    ax.grid(alpha=0.3)

    handles = [plt.Line2D([0], [0], marker='o', color='w',
                           label=get_display_name(m),
                           markerfacecolor=method_colors[m], markersize=12,
                           markeredgecolor='black', markeredgewidth=1.5)
               for m in subset['method'].tolist()]
    handles.append(plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2))
    handles.append(plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=2))
    m_labels = [get_display_name(m) for m in subset['method'].tolist()]
    m_labels += ['Target (90%)', 'Tolerance (88%)']

    ncol = max(1, (len(m_labels) + 2) // 3)
    ax.legend(handles, m_labels, loc='upper center', bbox_to_anchor=(0.5, -0.12),
              ncol=ncol, fontsize=LEGEND_SIZE)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)
    fname = f'coverage_width_tradeoff_{learner.replace(" ", "_").lower()}.png'
    plt.savefig(OUTPUT_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()


## 8. Outcome-Conditional Coverage

In [ ]:
basic_cov_cols = ['coverage_y0', 'coverage_y1', 'coverage_y2plus']
binned_cov_cols = ['coverage_y1_5', 'coverage_y6_10', 'coverage_y11_20',
                   'coverage_y21_50', 'coverage_y51plus']
available_basic = [c for c in basic_cov_cols if c in df_baseline.columns]
available_binned = [c for c in binned_cov_cols if c in df_baseline.columns]

print(f'Basic OC columns: {available_basic}')
print(f'Binned OC columns: {available_binned}')


In [ ]:
def create_oc_barplot(df_data, title_suffix, filename_suffix, methods_filter=None):
    if methods_filter is not None:
        df_data = df_data[df_data['method'].isin(methods_filter)]
    df_sub = df_data.dropna(subset=available_basic[:2]) if available_basic else df_data
    if len(df_sub) == 0:
        return

    dgps = df_sub['dgp_type'].unique()[:4]
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    legend_handles, legend_labels = None, None

    for idx, dgp in enumerate(dgps):
        ax = axes.flatten()[idx]
        subset = df_sub[df_sub['dgp_type'] == dgp]
        grp = subset.groupby('method')[available_basic].mean().reset_index()
        grp['display_name'] = grp['method'].apply(get_display_name)
        grp = grp.sort_values('display_name')

        x = np.arange(len(grp))
        width = 0.25
        colors = sns.color_palette('colorblind', 3)
        ax.bar(x - width, grp[available_basic[0]], width, label='Y=0', alpha=0.8, color=colors[0])
        if len(available_basic) > 1:
            ax.bar(x, grp[available_basic[1]], width, label='Y=1', alpha=0.8, color=colors[1])
        if len(available_basic) > 2:
            ax.bar(x + width, grp[available_basic[2]], width, label='Y>=2', alpha=0.8, color=colors[2])

        ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
        ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
        ax.set_xticks(x)
        ax.set_xticklabels(grp['display_name'], rotation=45, ha='right', fontsize=TICK_SIZE)
        ax.set_ylabel('Coverage')
        ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
        ax.set_ylim(0.5, 1.05)
        ax.tick_params(axis='y', labelsize=TICK_SIZE)

        if idx == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            legend_handles.append(plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2))
            legend_handles.append(plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5))
            legend_labels.extend(['Target (90%)', 'Tolerance (88%)'])

    fig.legend(legend_handles, legend_labels, loc='upper center', bbox_to_anchor=(0.5, 0.05),
               ncol=len(legend_labels), fontsize=LEGEND_SIZE)
    plt.suptitle(f'Outcome-Conditional Coverage {title_suffix}', fontsize=TITLE_SIZE+2, fontweight='bold')
    plt.tight_layout(rect=[0, 0.06, 1, 0.96])
    plt.savefig(OUTPUT_DIR / f'outcome_conditional_{filename_suffix}.png', dpi=150, bbox_inches='tight')
    plt.show()

if available_basic:
    create_oc_barplot(df_baseline, '(All Methods)', 'all_methods')
    cp_all = CP_METHODS + DGCP_METHODS
    create_oc_barplot(df_baseline, '(CP Methods)', 'cp_only', methods_filter=cp_all)
    create_oc_barplot(df_baseline, '(Benchmarks)', 'benchmarks_only', methods_filter=BENCHMARK_METHODS)


In [ ]:
# Binned outcome-conditional coverage
bin_labels = {
    'coverage_y1_5': 'Y in [1,5]',
    'coverage_y6_10': 'Y in [6,10]',
    'coverage_y11_20': 'Y in [11,20]',
    'coverage_y21_50': 'Y in [21,50]',
    'coverage_y51plus': 'Y >= 51',
}

if available_binned:
    df_binned = df_baseline.dropna(subset=[c for c in available_binned
                                             if c in df_baseline.columns][:1])
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    legend_handles, legend_labels = None, None

    for idx, dgp in enumerate(df_binned['dgp_type'].unique()[:4]):
        ax = axes.flatten()[idx]
        subset = df_binned[df_binned['dgp_type'] == dgp]
        cols_present = [c for c in available_binned
                        if c in subset.columns and subset[c].notna().any()]
        if not cols_present:
            ax.text(0.5, 0.5, 'No binned data', ha='center', va='center', transform=ax.transAxes)
            continue

        grp = subset.groupby('method')[cols_present].mean().reset_index()
        grp['display_name'] = grp['method'].apply(get_display_name)
        grp = grp.sort_values('display_name')

        x = np.arange(len(grp))
        n_bins = len(cols_present)
        bw = 0.8 / n_bins
        colors = sns.color_palette('viridis', n_bins)

        for i, col in enumerate(cols_present):
            offset = (i - n_bins / 2 + 0.5) * bw
            ax.bar(x + offset, grp[col], bw,
                   label=bin_labels.get(col, col), alpha=0.8, color=colors[i])

        ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
        ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
        ax.set_xticks(x)
        ax.set_xticklabels(grp['display_name'], rotation=45, ha='right', fontsize=TICK_SIZE)
        ax.set_ylabel('Coverage')
        ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
        ax.set_ylim(0.0, 1.05)
        ax.tick_params(axis='y', labelsize=TICK_SIZE)

        if idx == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            legend_handles.append(plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2))
            legend_handles.append(plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5))
            legend_labels.extend(['Target (90%)', 'Tolerance (88%)'])

    fig.legend(legend_handles, legend_labels, loc='upper center', bbox_to_anchor=(0.5, 0.05),
               ncol=min(len(legend_labels), 7), fontsize=LEGEND_SIZE)
    plt.suptitle('Binned Outcome-Conditional Coverage', fontsize=TITLE_SIZE+2, fontweight='bold')
    plt.tight_layout(rect=[0, 0.06, 1, 0.96])
    plt.savefig(OUTPUT_DIR / 'outcome_conditional_binned.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Coverage gap analysis
if 'coverage_gap' in df_baseline.columns:
    gap_summary = df_baseline.groupby('method')['coverage_gap'].agg(['mean', 'std']).reset_index()
    gap_summary['display_name'] = gap_summary['method'].apply(get_display_name)
    gap_summary = gap_summary.sort_values('mean', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['green' if g < 0.05 else 'orange' if g < 0.10 else 'red'
              for g in gap_summary['mean']]
    ax.barh(gap_summary['display_name'], gap_summary['mean'],
            xerr=gap_summary['std'], color=colors, alpha=0.7, capsize=3)
    ax.axvline(0.05, color='green', linestyle='--', linewidth=2, label='Good (<5%)')
    ax.axvline(0.10, color='orange', linestyle=':', linewidth=1.5, label='Moderate (<10%)')
    ax.set_xlabel('Coverage Gap (|Cov(Y=0) - Cov(Y=1)|)')
    ax.set_title('Coverage Gap Between Zero and Non-Zero Claims',
                 fontsize=TITLE_SIZE, fontweight='bold')
    ax.tick_params(axis='both', labelsize=TICK_SIZE)
    ax.legend(loc='lower right', fontsize=LEGEND_SIZE)
    ax.set_xlim(0, None)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'coverage_gap_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Outcome-conditional coverage heatmaps
all_oc_cols = ['coverage_y0', 'coverage_y1_5', 'coverage_y6_10',
               'coverage_y11_20', 'coverage_y21_50']
available_oc = [c for c in all_oc_cols if c in df_baseline.columns]

col_rename = {
    'coverage_y0': 'Y=0',
    'coverage_y1': 'Y=1',
    'coverage_y2plus': 'Y>=2',
    'coverage_y1_5': 'Y in [1,5]',
    'coverage_y6_10': 'Y in [6,10]',
    'coverage_y11_20': 'Y in [11,20]',
    'coverage_y21_50': 'Y in [21,50]',
    'coverage_y51plus': 'Y>=51',
}

cmap = LinearSegmentedColormap.from_list('coverage',
    [(0.8, 0.2, 0.2), (1, 1, 0.4), (0.2, 0.8, 0.2)])

def save_oc_heatmap(data, title_suffix, filename_suffix):
    if data.empty:
        return
    oc_pivot = data.groupby('method')[available_oc].mean()
    oc_pivot.index = [get_display_name(m) for m in oc_pivot.index]
    oc_pivot = oc_pivot.rename(columns=col_rename)

    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(oc_pivot, annot=True, fmt='.3f', cmap=cmap,
                vmin=0.70, vmax=0.98, ax=ax,
                cbar_kws={'label': 'Coverage'}, linewidths=0.5, linecolor='white')
    ax.set_title(f'Outcome-Conditional Coverage {title_suffix}',
                 fontsize=TITLE_SIZE, fontweight='bold')
    ax.set_ylabel('Method')
    ax.set_xlabel('Outcome Group')
    ax.tick_params(axis='both', labelsize=TICK_SIZE)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'oc_heatmap_{filename_suffix}.png', dpi=150, bbox_inches='tight')
    plt.show()

if available_oc:
    save_oc_heatmap(df_baseline, '(All DGPs)', 'overall')
    for dgp in df_baseline['dgp_type'].unique():
        save_oc_heatmap(df_baseline[df_baseline['dgp_type'] == dgp],
                        f'(DGP: {dgp.upper()})', dgp)

    # Summary statistics table
    oc_pivot = df_baseline.groupby('method')[available_oc].mean()
    oc_pivot.index = [get_display_name(m) for m in oc_pivot.index]
    oc_stats = pd.DataFrame({
        'Mean Coverage': oc_pivot.mean(axis=1),
        'Min Coverage': oc_pivot.min(axis=1),
        'Max Coverage': oc_pivot.max(axis=1),
        'Range': oc_pivot.max(axis=1) - oc_pivot.min(axis=1),
        'Std': oc_pivot.std(axis=1),
    }).round(4).sort_values('Range')
    display(oc_stats)


## 9. Sensitivity Analysis

### 9a. Zero Inflation Effect

In [ ]:
if 'zero_inflation' in df.columns:
    df_zi = apply_baseline_filter(df, exclude='zero_inflation')
    df_zi = df_zi[df_zi['dgp_type'].isin(['zip', 'hurdle'])]

    if len(df_zi) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 8))
        legend_handles, legend_labels = None, None

        for idx, dgp in enumerate(['zip', 'hurdle']):
            ax = axes[idx]
            subset = df_zi[df_zi['dgp_type'] == dgp]
            if len(subset) == 0:
                continue
            zi_summary = subset.groupby(['zero_inflation', 'method'])['coverage'].mean().unstack()
            zi_summary.columns = [get_display_name(m) for m in zi_summary.columns]
            zi_summary.plot(kind='line', marker='o', ax=ax, alpha=0.7, legend=False)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
            ax.set_xlabel('Zero Inflation Level')
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.set_ylim(0.7, 1.0)
            ax.tick_params(axis='both', labelsize=TICK_SIZE)

            if idx == 0:
                legend_handles, legend_labels = ax.get_legend_handles_labels()
                legend_handles += [
                    plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2),
                    plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5),
                ]
                legend_labels += ['Target (90%)', 'Tolerance (88%)']

        fig.legend(legend_handles, legend_labels, loc='upper center',
                   bbox_to_anchor=(0.5, 0.05), ncol=len(legend_labels), fontsize=LEGEND_SIZE)
        plt.tight_layout(rect=[0, 0.06, 1, 0.96])
        plt.savefig(OUTPUT_DIR / 'zero_inflation_effect.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9b. Sample Size Effect

In [ ]:
if 'n_obs' in df.columns:
    df_n = apply_baseline_filter(df, exclude='n_obs')

    if len(df_n) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        legend_handles, legend_labels = None, None

        for idx, dgp in enumerate(df_n['dgp_type'].unique()[:4]):
            ax = axes.flatten()[idx]
            subset = df_n[df_n['dgp_type'] == dgp]
            n_summary = subset.groupby(['n_obs', 'method'])['coverage'].mean().unstack()
            n_summary.columns = [get_display_name(m) for m in n_summary.columns]
            n_summary.plot(kind='line', marker='s', ax=ax, alpha=0.7, legend=False)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
            ax.set_xlabel('Sample Size (n_obs)')
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.set_ylim(0.7, 1.0)
            ax.tick_params(axis='both', labelsize=TICK_SIZE)

            if idx == 0:
                legend_handles, legend_labels = ax.get_legend_handles_labels()
                legend_handles += [
                    plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2),
                    plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5),
                ]
                legend_labels += ['Target (90%)', 'Tolerance (88%)']

        fig.legend(legend_handles, legend_labels, loc='upper center',
                   bbox_to_anchor=(0.5, 0.05), ncol=len(legend_labels), fontsize=LEGEND_SIZE)
        plt.tight_layout(rect=[0, 0.06, 1, 0.96])
        plt.savefig(OUTPUT_DIR / 'sample_size_effect.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9c. Covariate Dimension Effect

In [ ]:
if 'dim_x' in df.columns:
    df_dim = apply_baseline_filter(df, exclude='dim_x')

    if len(df_dim) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        legend_handles, legend_labels = None, None

        for idx, dgp in enumerate(df_dim['dgp_type'].unique()[:4]):
            ax = axes.flatten()[idx]
            subset = df_dim[df_dim['dgp_type'] == dgp]
            dim_summary = subset.groupby(['dim_x', 'method'])['coverage'].mean().unstack()
            dim_summary.columns = [get_display_name(m) for m in dim_summary.columns]
            dim_summary.plot(kind='line', marker='D', ax=ax, alpha=0.7, legend=False)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
            ax.set_xlabel('Covariate Dimension (dim_x)')
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.set_ylim(0.7, 1.0)
            ax.tick_params(axis='both', labelsize=TICK_SIZE)

            if idx == 0:
                legend_handles, legend_labels = ax.get_legend_handles_labels()
                legend_handles += [
                    plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2),
                    plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5),
                ]
                legend_labels += ['Target (90%)', 'Tolerance (88%)']

        fig.legend(legend_handles, legend_labels, loc='upper center',
                   bbox_to_anchor=(0.5, 0.05), ncol=len(legend_labels), fontsize=LEGEND_SIZE)
        plt.tight_layout(rect=[0, 0.06, 1, 0.96])
        plt.savefig(OUTPUT_DIR / 'covariate_dimension_effect.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9d. Correlation Strength Effect

In [ ]:
if 'correlation_strength' in df.columns:
    df_corr = apply_baseline_filter(df, exclude='correlation_strength')

    if len(df_corr) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        legend_handles, legend_labels = None, None

        for idx, dgp in enumerate(df_corr['dgp_type'].unique()[:4]):
            ax = axes.flatten()[idx]
            subset = df_corr[df_corr['dgp_type'] == dgp]
            corr_summary = subset.groupby(['correlation_strength', 'method'])['coverage'].mean().unstack()
            corr_summary.columns = [get_display_name(m) for m in corr_summary.columns]
            corr_summary.plot(kind='line', marker='^', ax=ax, alpha=0.7, legend=False)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
            ax.set_xlabel('Correlation Strength')
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.set_ylim(0.7, 1.0)
            ax.tick_params(axis='both', labelsize=TICK_SIZE)

            if idx == 0:
                legend_handles, legend_labels = ax.get_legend_handles_labels()
                legend_handles += [
                    plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2),
                    plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5),
                ]
                legend_labels += ['Target (90%)', 'Tolerance (88%)']

        fig.legend(legend_handles, legend_labels, loc='upper center',
                   bbox_to_anchor=(0.5, 0.05), ncol=len(legend_labels), fontsize=LEGEND_SIZE)
        plt.tight_layout(rect=[0, 0.06, 1, 0.96])
        plt.savefig(OUTPUT_DIR / 'correlation_strength_effect.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9e. Categorical Ratio Effect

In [ ]:
if 'categorical_ratio' in df.columns:
    df_cat = apply_baseline_filter(df, exclude='categorical_ratio')

    if len(df_cat) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        legend_handles, legend_labels = None, None

        for idx, dgp in enumerate(df_cat['dgp_type'].unique()[:4]):
            ax = axes.flatten()[idx]
            subset = df_cat[df_cat['dgp_type'] == dgp]
            cat_summary = subset.groupby(['categorical_ratio', 'method'])['coverage'].mean().unstack()
            cat_summary.columns = [get_display_name(m) for m in cat_summary.columns]
            cat_summary.plot(kind='line', marker='v', ax=ax, alpha=0.7, legend=False)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5)
            ax.set_xlabel('Categorical Ratio')
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.set_ylim(0.7, 1.0)
            ax.tick_params(axis='both', labelsize=TICK_SIZE)

            if idx == 0:
                legend_handles, legend_labels = ax.get_legend_handles_labels()
                legend_handles += [
                    plt.Line2D([0], [1], color='red', linestyle='--', linewidth=2),
                    plt.Line2D([0], [1], color='orange', linestyle=':', linewidth=1.5),
                ]
                legend_labels += ['Target (90%)', 'Tolerance (88%)']

        fig.legend(legend_handles, legend_labels, loc='upper center',
                   bbox_to_anchor=(0.5, 0.05), ncol=len(legend_labels), fontsize=LEGEND_SIZE)
        plt.tight_layout(rect=[0, 0.06, 1, 0.96])
        plt.savefig(OUTPUT_DIR / 'categorical_ratio_effect.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9f. Overdispersion Effect

In [ ]:
if 'overdispersion' in df.columns:
    df_od = apply_baseline_filter(df, exclude='overdispersion')
    df_od = df_od[df_od['dgp_type'] == 'negbin']

    if len(df_od) > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        od_summary = df_od.groupby(['overdispersion', 'method'])['coverage'].mean().unstack()
        od_summary.columns = [get_display_name(m) for m in od_summary.columns]
        od_summary.plot(kind='line', marker='o', ax=ax, alpha=0.7)
        ax.axhline(0.90, color='red', linestyle='--', linewidth=2, label='Target (90%)')
        ax.axhline(0.88, color='orange', linestyle=':', linewidth=1.5, label='Tolerance (88%)')
        ax.set_xlabel('Overdispersion Level')
        ax.set_ylabel('Coverage')
        ax.set_title('Overdispersion Effect (NegBin DGP)', fontsize=TITLE_SIZE, fontweight='bold')
        ax.set_ylim(0.7, 1.0)
        ax.legend(loc='lower right', fontsize=LEGEND_SIZE)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'overdispersion_effect.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9g. Interaction and Spatial Effects

In [ ]:
# Interaction effects
if 'interaction_effects' in df.columns:
    df_int = apply_baseline_filter(df, exclude='interaction_effects')

    if len(df_int) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        for idx, dgp in enumerate(df_int['dgp_type'].unique()[:4]):
            ax = axes.flatten()[idx]
            subset = df_int[df_int['dgp_type'] == dgp]
            for method in subset['method'].unique():
                m_data = subset[subset['method'] == method]
                for ie_val in sorted(m_data['interaction_effects'].unique()):
                    cov = m_data[m_data['interaction_effects'] == ie_val]['coverage'].mean()
                    ax.bar(get_display_name(method) + f' (int={ie_val})', cov, alpha=0.7)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.tick_params(axis='x', rotation=45, labelsize=TICK_SIZE)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'interaction_effects.png', dpi=150, bbox_inches='tight')
        plt.show()

# Spatial effects
if 'spatial_effects' in df.columns:
    df_sp = apply_baseline_filter(df, exclude='spatial_effects')

    if len(df_sp) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        for idx, dgp in enumerate(df_sp['dgp_type'].unique()[:4]):
            ax = axes.flatten()[idx]
            subset = df_sp[df_sp['dgp_type'] == dgp]
            for method in subset['method'].unique():
                m_data = subset[subset['method'] == method]
                for sp_val in sorted(m_data['spatial_effects'].unique()):
                    cov = m_data[m_data['spatial_effects'] == sp_val]['coverage'].mean()
                    ax.bar(get_display_name(method) + f' (spatial={sp_val})', cov, alpha=0.7)
            ax.axhline(0.90, color='red', linestyle='--', linewidth=2)
            ax.set_ylabel('Coverage')
            ax.set_title(f'DGP: {dgp.upper()}', fontsize=TITLE_SIZE, fontweight='bold')
            ax.tick_params(axis='x', rotation=45, labelsize=TICK_SIZE)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'spatial_effects.png', dpi=150, bbox_inches='tight')
        plt.show()


### 9h. Sensitivity Analysis Overview

In [ ]:
sensitivity_params = {
    'n_obs': 'Sample Size',
    'dim_x': 'Covariate Dimension',
    'correlation_strength': 'Correlation Strength',
    'categorical_ratio': 'Categorical Ratio',
    'zero_inflation': 'Zero Inflation',
    'overdispersion': 'Overdispersion',
}

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for idx, (param, label) in enumerate(sensitivity_params.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]
    if param not in df.columns:
        ax.text(0.5, 0.5, f'{param} not in data', ha='center', va='center', transform=ax.transAxes)
        continue

    df_var = apply_baseline_filter(df, exclude=param)
    if param == 'zero_inflation':
        df_var = df_var[df_var['dgp_type'].isin(['zip', 'hurdle'])]
    elif param == 'overdispersion':
        df_var = df_var[df_var['dgp_type'] == 'negbin']

    if len(df_var) == 0:
        ax.text(0.5, 0.5, f'No data for {param}', ha='center', va='center', transform=ax.transAxes)
        continue

    var_summary = df_var.groupby(param).agg({
        'coverage': 'mean',
        'mean_width': 'mean',
    }).reset_index()

    ax.plot(var_summary[param], var_summary['coverage'], 'o-', label='Coverage', linewidth=2)
    ax.axhline(0.90, color='red', linestyle='--', linewidth=1.5)
    ax.axhline(0.88, color='orange', linestyle=':', linewidth=1)
    ax.set_xlabel(label)
    ax.set_ylabel('Coverage')
    ax.set_title(f'Sensitivity: {label}', fontsize=TITLE_SIZE, fontweight='bold')
    ax.tick_params(axis='both', labelsize=TICK_SIZE)
    ax.grid(alpha=0.3)

for idx in range(len(sensitivity_params), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sensitivity_overview.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Method Rankings

In [ ]:
rankings = compute_method_rankings(df_baseline)
rankings['display_name'] = rankings['method'].apply(get_display_name)

# Combined ranking (average across all DGP/learner combos)
overall = rankings.groupby('method').agg({
    'rank_coverage': 'mean',
    'rank_width': 'mean',
    'rank_interval_score': 'mean',
    'rank_gap': 'mean',
    'rank_combined': 'mean',
}).reset_index().sort_values('rank_combined')
overall['display_name'] = overall['method'].apply(get_display_name)
display(overall)

# Top 3 by combined rank
top3 = overall.head(3)
print('Top 3 methods (combined ranking):')
for _, row in top3.iterrows():
    print(f'  {row["display_name"]}: combined rank = {row["rank_combined"]:.2f}')


In [ ]:
# Rankings heatmap
fig, ax = plt.subplots(figsize=(14, 10))

rank_pivot = rankings.pivot_table(
    values='rank_combined', index='method',
    columns=['dgp_type', 'learner'], aggfunc='mean')
rank_pivot.index = [get_display_name(m) for m in rank_pivot.index]

sns.heatmap(rank_pivot, annot=True, fmt='.1f', cmap='YlOrRd_r',
            vmin=1, ax=ax, cbar_kws={'label': 'Combined Rank'})
ax.set_title('Method Rankings by DGP and Learner', fontsize=TITLE_SIZE, fontweight='bold')
ax.set_ylabel('Method')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'method_rankings_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Publication table
latex = generate_latex_table(df_baseline, OUTPUT_DIR)
print(latex)


## 11. Key Findings

In [ ]:
print('KEY FINDINGS')
print('=' * 60)

# Coverage summary
coverage_by_method = df_baseline.groupby('method')['coverage'].mean().sort_values(ascending=False)
valid_methods = coverage_by_method[coverage_by_method >= 0.88]
print(f'Methods achieving >= 88% coverage: {len(valid_methods)} / {len(coverage_by_method)}')
print(f'Best coverage: {get_display_name(coverage_by_method.index[0])} ({coverage_by_method.iloc[0]:.4f})')

# Width summary
width_by_method = df_baseline.groupby('method')['mean_width'].mean().sort_values(ascending=True)
print(f'Narrowest intervals: {get_display_name(width_by_method.index[0])} (width={width_by_method.iloc[0]:.2f})')

# Best overall from rankings
print(f'Best overall (combined rank): {overall.iloc[0]["display_name"]} (rank={overall.iloc[0]["rank_combined"]:.2f})')

# Outcome-conditional
if 'coverage_gap' in df_baseline.columns:
    min_gap_method = df_baseline.groupby('method')['coverage_gap'].mean().idxmin()
    min_gap = df_baseline.groupby('method')['coverage_gap'].mean().min()
    print(f'Most uniform OC coverage: {get_display_name(min_gap_method)} (gap={min_gap:.4f})')


In [ ]:
# Coverage heatmap: method x (dgp, learner)
pivot = df_baseline.pivot_table(
    values='coverage', index='method',
    columns=['dgp_type', 'learner'], aggfunc='mean')

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='cividis',
            vmin=0.75, vmax=0.95, ax=ax,
            cbar_kws={'label': 'Coverage'})
ax.set_title('Coverage Heatmap: Method x (DGP, Learner)', fontsize=TITLE_SIZE, fontweight='bold')
ax.set_ylabel('Method')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coverage_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
